# Análise de Localização — Unitree Go2

**Disciplina:** Ciência de Dados — FEI Mestrado  
**Descrição:** Análise da qualidade de localização do RTABMAP utilizando uma ou duas câmeras no robô Go2.

---

### Fontes de Dados
| Arquivo | Descrição |
|---------|-----------|
| `localization_log.csv` | Métricas por atualização dos tópicos `/rtabmap/info` e `/localization_pose` (inliers, covariância, pose, etc.) |
| `plan_log.csv` | Poses do caminho planejado pelo tópico `/plan`, agrupadas por `plan_id` |

### Seções
1. **Carregamento e Processamento dos Dados** — leitura dos arquivos CSV e pré-processamento
2. **Visualização dos Caminhos** — caminho planejado vs trajetória real do robô
3. **Qualidade da Localização** — inliers, razão de hipótese e covariância ao longo do tempo

### 1. Carregamento e Processamento dos Dados

Os dados foram coletados em 20 execuções do robô Go2, divididas em dois grupos:
- **Runs 1–10:** robô equipado com **duas câmeras**
- **Runs 11–20:** robô equipado com **uma câmera**

Cada run possui dois arquivos de log: um com as métricas de localização e outro com o caminho planejado pelo stack de navegação NAV2.

In [377]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from scipy.stats.mstats import winsorize
from plotly.subplots import make_subplots

DEBUG = False  # Set to True to print intermediate values


logs_dir = Path("localization_analysis/data/logs")

loc_csv_paths = sorted(logs_dir.glob("logger_csv_*/localization_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_loc_df    = [pd.read_csv(loc_path) for loc_path in loc_csv_paths] 

plan_csv_paths = sorted(logs_dir.glob("logger_csv_*/plan_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_plan_df    = [pd.read_csv(plan_path) for plan_path in plan_csv_paths] 

In [378]:
print(f' Localization Dataframes: {len(runs_loc_df)}\n Plan Dataframes {len(runs_plan_df)}') # Number of DF

 Localization Dataframes: 20
 Plan Dataframes 20


In [379]:
# Take the first path that NAV2 stack calculated (the plan_id = 1)
plans_df = [df[df['plan_id'] == df['plan_id'].min()] for df in runs_plan_df]
plans_df[0]

,plan_id,pose_index,x,y
0,1,0,-1.9519,1.0049
1,1,1,-1.9169,0.9549
2,1,2,-1.8827,0.9049
3,1,3,-1.8499,0.8549
4,1,4,-1.8184,0.8049
...,...,...,...,...
125,1,125,1.7117,1.5549
126,1,126,1.7184,1.6049
127,1,127,1.7269,1.6549
128,1,128,1.7370,1.7049


In [380]:
# Drop rows where pose was not yet received (NaN)
paths_df = [df.dropna(subset=['pos_x', 'pos_y']) for df in runs_loc_df]
paths_df

[    timestamp_sec camera_mode  node_id  inliers  matches  inlier_ratio  \
 0    1.776904e+09      single    19302        0        0        0.0000   
 1    1.776904e+09      single    19304        0        0        0.0000   
 2    1.776904e+09      single    19306        3       33        0.0066   
 3    1.776904e+09      single    19307        0        0        0.0000   
 4    1.776904e+09      single    19308       26      160        0.0582   
 5    1.776904e+09      single    19309        0        0        0.0000   
 6    1.776904e+09      single    19310        0        0        0.0000   
 7    1.776904e+09      single    19311       34      157        0.0787   
 8    1.776904e+09      single    19312       27      129        0.0619   
 9    1.776904e+09      single    19313        0        0        0.0000   
 10   1.776904e+09      single    19314        0        0        0.0000   
 11   1.776904e+09      single    19315       46      159        0.1117   
 12   1.776904e+09      s

### 2. Visualização dos Caminhos

Comparação entre o caminho planejado pelo NAV2 e a trajetória real percorrida pelo robô em cada execução.

In [381]:
n_runs = len(paths_df)
cols = 2
rows = math.ceil(n_runs / cols)
titles = [f'Run {i+1}' for i in range(n_runs)]

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=titles,
                    shared_yaxes=False)

for i, (actual_df, last_plan) in enumerate(zip(paths_df, plans_df)):
    row = i // cols + 1
    col = i % cols + 1

    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Planned path',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=2),
        showlegend=(i == 0),
        legendgroup=f'run{i+1}'
    ), row=row, col=col)

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(
    title='Planned vs Actual Path',
    hovermode='closest',
    height=500 * rows,
    width=900
)
fig.show()


#### 2.1 Normalização e Mediana dos Caminhos

Para comparar os caminhos entre diferentes runs, é necessário normalizá-los. Cada execução possui um número diferente de amostras e uma velocidade diferente, então não é possível comparar ponto a ponto diretamente.

A abordagem utilizada é a **parametrização por comprimento de arco**:
1. **Winsorize** — remove poses extremas 
2. **Comprimento de arco normalizado** — mapeia cada ponto para `[0, 1]` proporcionalmente à distância percorrida
3. **Interpolação** — reamostrar todos os caminhos para `n_points` pontos uniformes

Com isso, todos os caminhos ficam no mesmo espaço e é possível calcular a **mediana ponto a ponto** para os dfs de 1 e 2 câmeras.

##### 2.1.1 Caminhos Planejados para duas câmeras (Plans)

Aplicamos a normalização nos caminhos planejados pelo NAV2. Como os planos são gerados com amostras equidistantes, o efeito da interpolação é sutil, mas é necessário para manter o mesmo pipeline dos caminhos reais.

In [382]:
def arc_length_parametrize(pos_x, pos_y):
    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0) # diff in each point (x)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])

    last_element = arc[-1]
    arc_norm = arc / last_element

    if DEBUG:
        print(f"[arc_length_parametrize]")
        print(f"  n_points      : {len(pos_x)}")
        print(f"  total length  : {last_element:.4f} m")
        print(f"  arc[:5]  : {arc[:5]}")
        print(f"  arc[-5:] : {arc[-5:]}")
        print(f"  arc_norm[:5]  : {arc_norm[:5]}")
        print(f"  arc_norm[-5:] : {arc_norm[-5:]}")

    return arc_norm

def plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Interpolation check'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(u, pos_x, 'o', t, fx(t), '-')
    ax1.set_title('X vs arc_norm')
    ax1.set_xlabel('arc_norm (0→1)')
    ax2.plot(u, pos_y, 'o', t, fy(t), '-')
    ax2.set_title('Y vs arc_norm')
    ax2.set_xlabel('arc_norm (0→1)')
    plt.suptitle(f'[DEBUG] {title}')
    plt.tight_layout()
    plt.show()

def winsorize_and_resample(df, x_col='pos_x', y_col='pos_y', limits=(0.0075, 0.015), n_points=500, idx=None):
    if idx is not None and idx >= 10:
        if DEBUG:
            print(f"[winsorize_and_resample — plans] skipped run {idx} (single camera — will process later)")
        return None

    pos_x = np.array(winsorize(df[x_col], limits=limits))
    pos_y = np.array(winsorize(df[y_col], limits=limits))

    if DEBUG:
        print(f"[winsorize_and_resample — plans]")
        print(f"  input rows    : {len(df)}")
        print(f"  limits        : {limits}")
        print(f"  pos_x range   : [{pos_x.min():.4f}, {pos_x.max():.4f}]")
        print(f"  pos_y range   : [{pos_y.min():.4f}, {pos_y.max():.4f}]")

    u = arc_length_parametrize(pos_x, pos_y)

    t = np.linspace(0, 1, n_points)
    fx = interp1d(u, pos_x, kind='linear')
    fy = interp1d(u, pos_y, kind='linear')

    # If you want to plot the graph (X,Arc_Param)
    if DEBUG:
        print(f"  resampled to  : {n_points} points")
        plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Plans')

    return fx(t), fy(t)

# Plan paths (use x/y columns)
plans_resampled_2cam = [winsorize_and_resample(df, x_col='x', y_col='y', idx=i) for i, df in enumerate(plans_df)]

plans_xs_2cam = np.array([p[0] for p in plans_resampled_2cam if p is not None])
plans_ys_2cam = np.array([p[1] for p in plans_resampled_2cam if p is not None])
median_plan_x_2cam = np.median(plans_xs_2cam, axis=0)
median_plan_y_2cam = np.median(plans_ys_2cam, axis=0)

**Verificação — Planos 2 câmeras:** comparação entre dados brutos e após reamostagem por comprimento de arco.

In [383]:
_df = plans_df[0]
_raw_x = _df['x'].values
_raw_y = _df['y'].values

_wins_x = np.array(winsorize(_df['x'], limits=(0.005, 0.013)))
_wins_y = np.array(winsorize(_df['y'], limits=(0.013, 0.02)))

_interp_x, _interp_y = plans_resampled_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Plano 1 — Etapas de Processamento', hovermode='closest', width=1300)
fig.show()

**Planos 2 câmeras — todos os planos reamostrados** com mediana calculada ponto a ponto.

In [384]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Median plan',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Plans + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

##### 2.1.2 Trajetórias Reais para duas câmeras (Paths)

Aplicamos o mesmo pipeline nos caminhos reais percorridos pelo robô. Aqui o winsorize é mais importante: o robô pode ter poses ruins no início ou fim da execução.

In [385]:
# Trajetórias reais — duas câmeras (winsorize + interpolação)
def _process_path_winsorize(df, n_points=500):
    pos_x = np.array(winsorize(df['pos_x'], limits=(0.0, 0.00)))
    pos_y = np.array(winsorize(df['pos_y'], limits=(0.0, 0.00)))
    u = arc_length_parametrize(pos_x, pos_y)
    t = np.linspace(0, 1, n_points)
    return interp1d(u, pos_x)(t), interp1d(u, pos_y)(t)

paths_winsorized_2cam = [_process_path_winsorize(df) for df in paths_df[:10]]
paths_xs_winsorized_2cam = np.array([p[0] for p in paths_winsorized_2cam])
paths_ys_winsorized_2cam = np.array([p[1] for p in paths_winsorized_2cam])
median_path_winsorized_x_2cam = np.median(paths_xs_winsorized_2cam, axis=0)
median_path_winsorized_y_2cam = np.median(paths_ys_winsorized_2cam, axis=0)


**Verificação — Trajetórias 2 câmeras (winsorize):** comparação entre sem processamento, winsorize e após interpolação.

In [386]:
_df = paths_df[0]
_raw_x = _df['pos_x'].values
_raw_y = _df['pos_y'].values

_wins_x = np.array(winsorize(_df['pos_x'], limits=(0.01, 0.00)))
_wins_y = np.array(winsorize(_df['pos_y'], limits=(0.0, 0.05)))

_interp_x, _interp_y = paths_winsorized_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Trajetória 1 — Etapas de Processamento', hovermode='closest', width=1300)
fig.show()

**Trajetórias 2 câmeras (winsorize) — todas as runs** com mediana calculada.

In [387]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in paths_winsorized_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

**Duas câmeras — mediana final:** sobreposição da mediana do plano calculado com a mediana da trajetória real.

In [388]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Duas Câmeras — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
fig.show()

##### MSE — Duas Câmeras

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados**. Um único número que resume o quanto o robô desviou do plano em média ao longo do percurso.

In [389]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_2cam, median_path_winsorized_x_2cam)
mse_y = mean_squared_error(median_plan_y_2cam, median_path_winsorized_y_2cam)
mse_2cam  = (mse_x + mse_y) / 2
rmse_2cam = np.sqrt(mse_2cam)
print(f'MSE  (mediana trajetória vs mediana plano) — duas câmeras: {mse_2cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — duas câmeras: {rmse_2cam:.4f} m')


MSE  (mediana trajetória vs mediana plano) — duas câmeras: 0.012410 m²
RMSE (mediana trajetória vs mediana plano) — duas câmeras: 0.1114 m


#### 2.2 Uma Câmera (Runs 11–20)

Repetimos o mesmo pipeline para as 10 execuções com uma câmera. As funções são idênticas — apenas o conjunto de dados muda.

##### 2.2.1 Caminhos Planejados

In [390]:
plans_resampled_1cam = [winsorize_and_resample(df, x_col='x', y_col='y') for df in plans_df[10:]]

plans_xs_1cam = np.array([p[0] for p in plans_resampled_1cam if p is not None])
plans_ys_1cam = np.array([p[1] for p in plans_resampled_1cam if p is not None])
median_plan_x_1cam = np.median(plans_xs_1cam, axis=0)
median_plan_y_1cam = np.median(plans_ys_1cam, axis=0)

In [391]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name='Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='plans', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Mediana dos planos',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Planos — Uma Câmera + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

##### 2.2.2 Trajetórias Reais para uma câmera (Paths)

Aplicamos exatamente o mesmo pipeline das duas câmeras (winsorize + interpolação por comprimento de arco). Anteriormente esta seção usava um *clip pela geometria do plano* como alternativa, mas a nova coleta de dados está limpa o suficiente para que o winsorize sozinho seja suficiente.

In [392]:
# Trajetórias reais — uma câmera (winsorize + interpolação)
paths_winsorized_1cam = [_process_path_winsorize(df) for df in paths_df[10:]]
paths_xs_winsorized_1cam = np.array([p[0] for p in paths_winsorized_1cam])
paths_ys_winsorized_1cam = np.array([p[1] for p in paths_winsorized_1cam])
median_path_winsorized_x_1cam = np.median(paths_xs_winsorized_1cam, axis=0)
median_path_winsorized_y_1cam = np.median(paths_ys_winsorized_1cam, axis=0)


Verificação visual da interpolação: o gráfico abaixo mostra todos os caminhos e a mediana calculada a partir das etapas de preprocessamentos citadas anteriormente.

In [393]:
fig = go.Figure()

for i, (rx, ry) in enumerate(p for p in paths_winsorized_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name='Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana das trajetórias',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Trajetórias — Uma Câmera (Winsorize) + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

Por fim será demonstrado gráficamente a junção das medianas tanto do caminho calculado, quanto o caminho feito.

In [394]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Uma Câmera — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
fig.show()

##### MSE — Uma Câmera

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados** para as runs com uma câmera.

In [395]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_1cam, median_path_winsorized_x_1cam)
mse_y = mean_squared_error(median_plan_y_1cam, median_path_winsorized_y_1cam)
mse_1cam  = (mse_x + mse_y) / 2
rmse_1cam = np.sqrt(mse_1cam)
print(f'MSE  (mediana trajetória vs mediana plano) — uma câmera: {mse_1cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — uma câmera: {rmse_1cam:.4f} m')

MSE  (mediana trajetória vs mediana plano) — uma câmera: 0.011882 m²
RMSE (mediana trajetória vs mediana plano) — uma câmera: 0.1090 m


#### 2.3 Comparação entre Grupos

Análise comparativa entre os dois grupos de execução — robô com duas câmeras (runs 1–10) e com uma câmera (runs 11–20) — dividida em duas etapas: análise gráfica das trajetórias e análise quantitativa via MSE/RMSE.

##### 2.3.1 Análise Gráfica

Comparação visual das trajetórias medianas dos dois grupos em relação ao plano mediano global.

In [396]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Duas Câmeras', 'Uma Câmera'])

# Duas câmeras
fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='royalblue', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=1)

# Uma câmera
fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='tomato', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=2)

fig.update_yaxes(scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', row=1, col=1)
fig.update_layout(
    title='Comparação: Mediana do Plano vs Mediana da Trajetória',
    hovermode='closest', width=1100
)
fig.show()

Para a comparação final, calcula-se a **mediana global do caminho planejado** — combinando os planos de ambos os grupos (duas e uma câmera). Com isso é possível visualizar num único gráfico o quanto cada grupo desviou em relação ao mesmo plano de referência.

In [397]:
# Mediana global do plano (une os dois grupos)
all_plans_xs = np.vstack([plans_xs_2cam, plans_xs_1cam])
all_plans_ys = np.vstack([plans_ys_2cam, plans_ys_1cam])
median_plan_x_global = np.median(all_plans_xs, axis=0)
median_plan_y_global = np.median(all_plans_ys, axis=0)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_global, y=median_plan_y_global,
    mode='lines', name='Plano mediano (global)',
    line=dict(color='gray', dash='dash', width=2)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Mediana trajetória — duas câmeras',
    line=dict(color='royalblue', width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana trajetória — uma câmera',
    line=dict(color='tomato', width=2.5)
))

fig.update_layout(
    title='Plano Mediano Global vs Medianas das Trajetórias',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

Como podemos ver, os dois caminhos ficaram bem parecidos, o que acabei ficando um pouco surpreso. Uma análise que pode nos dar uma informação mais precisa é o MSE e RMSE que calculamos anteriormente para os dois casos, esses valores quantificam exatamente o desvio médio de cada grupo em relação ao plano, independentemente da similaridade visual.

In [398]:
print('=' * 45)
print(f'  MSE  — duas câmeras : {mse_2cam:.6f} m²')
print(f'  RMSE — duas câmeras : {rmse_2cam:.4f} m')
print('-' * 45)
print(f'  MSE  — uma câmera   : {mse_1cam:.6f} m²')
print(f'  RMSE — uma câmera   : {rmse_1cam:.4f} m')
print('=' * 45)
diff_rmse = abs(rmse_2cam - rmse_1cam)
better = 'duas câmeras' if rmse_2cam < rmse_1cam else 'uma câmera'
print(f'  Diferença RMSE      : {diff_rmse:.4f} m')
print(f'  Grupo mais preciso  : {better}')
print('=' * 45)

  MSE  — duas câmeras : 0.012410 m²
  RMSE — duas câmeras : 0.1114 m
---------------------------------------------
  MSE  — uma câmera   : 0.011882 m²
  RMSE — uma câmera   : 0.1090 m
  Diferença RMSE      : 0.0024 m
  Grupo mais preciso  : uma câmera


##### 2.3.2 Análise Quantitativa — MSE e RMSE

O MSE e RMSE medem numericamente o desvio médio de cada grupo em relação ao plano. O boxplot complementa mostrando a dispersão dos erros entre as runs individuais — revelando se o desvio é consistente ou se há execuções muito discrepantes.

In [399]:
# RMSE de cada run individualmente vs plano mediano global
rmse_runs_2cam, runs_2cam = [], []
for i, (rx, ry) in enumerate(p for p in paths_winsorized_2cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_2cam.append(np.sqrt(np.mean(d)))
    runs_2cam.append(i + 1)        # runs 1..10

rmse_runs_1cam, runs_1cam = [], []
for i, (rx, ry) in enumerate(p for p in paths_winsorized_1cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_1cam.append(np.sqrt(np.mean(d)))
    runs_1cam.append(i + 11)       # runs 11..20

# Detecta outliers via regra de Tukey (Q1 - 1.5·IQR, Q3 + 1.5·IQR) — mesmo critério do boxplot
def _detect_outliers(rmses, run_ids, label):
    arr = np.asarray(rmses)
    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    outs = [(rid, v) for rid, v in zip(run_ids, rmses) if v < lo or v > hi]
    print(f'{label} — Q1={q1:.4f}  Q3={q3:.4f}  IQR={iqr:.4f}  fences=[{lo:.4f}, {hi:.4f}]')
    if outs:
        print(f'  Outliers ({len(outs)}):')
        for rid, v in sorted(outs, key=lambda x: -x[1]):
            print(f'    Run {rid:2d}: RMSE = {v:.4f} m')
    else:
        print('  Nenhum outlier.')

_detect_outliers(rmse_runs_2cam, runs_2cam, 'Duas câmeras')
_detect_outliers(rmse_runs_1cam, runs_1cam, 'Uma câmera ')

fig = go.Figure()
fig.add_trace(go.Box(
    y=rmse_runs_2cam, name='Duas câmeras',
    marker_color='royalblue', boxpoints='all', jitter=0.3, pointpos=-1.5,
    text=[f'Run {r}' for r in runs_2cam],
    hovertemplate='%{text}<br>RMSE = %{y:.4f} m<extra></extra>',
))
fig.add_trace(go.Box(
    y=rmse_runs_1cam, name='Uma câmera',
    marker_color='tomato', boxpoints='all', jitter=0.3, pointpos=-1.5,
    text=[f'Run {r}' for r in runs_1cam],
    hovertemplate='%{text}<br>RMSE = %{y:.4f} m<extra></extra>',
))
fig.update_layout(
    title='Dispersão do RMSE por Run — Duas Câmeras vs Uma Câmera',
    yaxis_title='RMSE (m)', hovermode='closest'
)
fig.show()

Duas câmeras — Q1=0.1498  Q3=0.1917  IQR=0.0418  fences=[0.0870, 0.2544]
  Outliers (1):
    Run  6: RMSE = 0.2775 m
Uma câmera  — Q1=0.1579  Q3=0.1665  IQR=0.0086  fences=[0.1450, 0.1793]
  Outliers (1):
    Run 11: RMSE = 0.1940 m


**Discussão - boxplot e conclusão sobre a equivalência das trajetórias**

O boxplot é especialmente útil aqui porque dá ao mesmo tempo uma visão da **distribuição do RMSE por run** (mediana, IQR e variação dos valores) e expõe explicitamente os **outliers** — pontos que ficam fora das cercas de Tukey (Q1 − 1.5·IQR, Q3 + 1.5·IQR).

Olhando os pontos rotulados, fica claro que existe **um outlier puxando o RMSE de duas câmeras para cima: a Run 6**. Ao inspecionar esse run em particular, percebe-se que ela teve uma falha de processamento — a localização só começou a ser contabilizada depois de algum tempo do início da execução, deixando uma faixa inicial sem dados e penalizando o RMSE médio dessa run especificamente. Não é, portanto, um reflexo da qualidade do sistema com duas câmeras, e sim de um problema isolado de coleta.

Mesmo com a Run 6 inflando o grupo de duas câmeras, o RMSE final ficou **muito próximo entre os dois modos** — diferença de apenas **0.0024 m** (ver a comparação MSE/RMSE acima). Combinando essa proximidade quantitativa com a sobreposição visual das medianas das trajetórias na análise gráfica, **conclui-se que o desempenho de localização com uma e com duas câmeras foi equivalente** neste benchmark.

A análise gráfica e quantitativa (MSE/RMSE) das trajetórias indicou desempenho **equivalente** entre os dois modos. No entanto, "equivalência na trajetória final" não garante que os dois modos cheguem lá da mesma forma, é possível que um deles esteja se localizando com mais incerteza, fazendo mais loop closures, ou consumindo mais recursos para entregar a mesma qualidade de caminho.

Por isso, a análise continua a seguir com foco nas **métricas internas de localização do RTABMAP** (`inlier_ratio`, `hypothesis_ratio`, `cov_pos_trace`, `detection_time_ms`, etc.), buscando identificar diferenças entre o uso de uma e duas câmeras que não aparecem na análise gráfica e no RMSE.

### 3. Qualidade da Localização

Análise das métricas internas do RTABMAP para cada grupo. Os dados são agrupados por câmera e comparados via boxplot (quando possível) para revelar diferenças na qualidade de localização visual, confiança da estimativa e custo computacional. As métricas de localização disponíveis são:

**Qualidade do casamento visual (RANSAC / odometria visual):**
- **`inliers`** — número de correspondências de features consideradas geometricamente consistentes pelo RANSAC ao estimar a transformação entre frames. Valores altos indicam que a cena tem textura suficiente e que o casamento visual foi bem-sucedido.
- **`matches`** — total de correspondências de features encontradas antes da filtragem geométrica. Reflete a riqueza visual da cena.
- **`inlier_ratio`** = `inliers / matches` — fração de correspondências válidas. É o melhor indicador *normalizado* da qualidade do casamento: valores baixos sugerem cenas pouco texturizadas, oclusões ou movimento brusco.

**Confiança do reconhecimento de lugar (loop closure):**
- **`hypothesis_ratio`** — razão entre a melhor hipótese de loop closure e o limiar de aceitação. Valores == 1 implicam que o RTABMAP fechou o laço, possuindo grande confiança do lugar; valores baixos indicam que o reconhecimento de lugar não está confiante.
- **`loop_closure_id`** — id do nó com o qual ocorreu o fechamento de laço (`-1` quando não houve). Útil para contar eventos de relocalização ao longo da trajetória.

**Incerteza da pose estimada:**
- **`cov_xx`, `cov_yy`, `cov_yaw`** — variâncias diagonais da matriz de covariância da pose. Quanto menores, mais confiante está o filtro sobre a estimativa.
- **`cov_pos_trace`** = `cov_xx + cov_yy` — traço da submatriz de posição; resume em um único escalar a incerteza translacional da estimativa.

**Custo computacional e gerenciamento de memória:**
- **`detection_time_ms`** — tempo gasto pelo módulo de detecção de loop closure por atualização.
- **`total_time_ms`** — tempo total de processamento do RTABMAP por atualização (detecção + manutenção do grafo + memória).
- **`wm_size`** — tamanho da *Working Memory* (número de nós ativos do grafo). Cresce com a exploração e impacta diretamente o custo computacional.

#### 3.1 Preparação dos Dados

Agregamos todos os logs de localização separados por grupo para facilitar a comparação.

In [400]:
# Concatena todos os runs de cada grupo com label
loc_2cam_raw = pd.concat(
    [df.assign(run=i+1) for i, df in enumerate(runs_loc_df[:10])],
    ignore_index=True
)
loc_1cam_raw = pd.concat(
    [df.assign(run=i+11) for i, df in enumerate(runs_loc_df[10:])],
    ignore_index=True
)

Plot do histograma para ver a distribuição dos dados e ter uma ideia de como estão, se possuem outliers.

In [401]:
metrics = ['inlier_ratio', 'hypothesis_ratio', 'cov_pos_trace', 'detection_time_ms']

fig = make_subplots(
    rows=len(metrics), cols=2,
    column_titles=['Duas Câmeras', 'Uma Câmera'],
    row_titles=metrics,
    vertical_spacing=0.06
)

for row, metric in enumerate(metrics, start=1):
    fig.add_trace(go.Histogram(
        x=loc_2cam_raw[metric], name=metric,
        marker_color='royalblue', opacity=0.7,
        showlegend=False
    ), row=row, col=1)
    fig.add_trace(go.Histogram(
        x=loc_1cam_raw[metric], name=metric,
        marker_color='tomato', opacity=0.7,
        showlegend=False
    ), row=row, col=2)

fig.update_layout(
    title='Distribuição das Métricas — Duas Câmeras vs Uma Câmera',
    height=300 * len(metrics), width=900,
    hovermode='closest'
)
fig.show()

Observando os histogramas, é nítida a presença de outliers apenas uma métrica agora: `inlier_ratio`. A seguir é feita uma análise para tentar explicar esses outliers e avaliar se é possível removê-los ou se representam alguma informação relevante sobre a localização durante a trajetória do robô.

**Hipótese 1 — `inlier_ratio = 0`:** correspondem a momentos em que há perda de frames das câmeras ou ao início da run, com o robô ainda parado, em que o RTABMAP ainda não processa correspondências visuais.

Também é perceptível que a `covariância > 100` não está presente nesse *dataset*.

Para verificar essa  hipótese, plotamos abaixo `inlier_ratio` em função do tempo relativo ao início de cada run.

In [402]:
# Análise temporal dos outliers — usa os dados brutos (sem filtro) para
# revelar EM QUE INSTANTE da trajetória os valores extremos aparecem.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _concat_with_rel_time(runs_subset, run_offset):
    parts = []
    for i, df in enumerate(runs_subset):
        d = df.copy()
        d['run']   = i + run_offset
        d['t_rel'] = d['timestamp_sec'] - d['timestamp_sec'].min()
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

_loc_2cam_t = _concat_with_rel_time(runs_loc_df[:10], 1)
_loc_1cam_t = _concat_with_rel_time(runs_loc_df[10:], 11)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'inlier_ratio — Duas Câmeras', 'inlier_ratio — Uma Câmera'
    ),
    vertical_spacing=0.13, horizontal_spacing=0.08,
)

for df_g, color, col in [(_loc_2cam_t, 'royalblue', 1),
                         (_loc_1cam_t, 'tomato',    2)]:
    for run_id, df_run in df_g.groupby('run'):
        fig.add_trace(go.Scatter(
            x=df_run['t_rel'], y=df_run['inlier_ratio'],
            mode='markers', marker=dict(color=color, size=3),
            opacity=0.45, showlegend=False,
            hovertemplate=f'run {run_id}<br>t=%{{x:.1f}}s<br>inlier_ratio=%{{y:.2f}}<extra></extra>'
        ), row=1, col=col)

for col in (1, 2):
    fig.update_xaxes(title_text='Tempo relativo ao início da run (s)', row=1, col=col)
fig.update_yaxes(title_text='inlier_ratio',        row=1, col=1)

fig.update_layout(
    title='Outliers ao longo da trajetória — inlier_ratio e cov_pos_trace vs. tempo',
    height=720, width=1100, hovermode='closest',
)
fig.show()

In [403]:
# Remove amostras sem localização válida:
#   1) sentinel cov_xx=9999 / cov_yy=9999  → robô no docking ou antes do primeiro loop closure
#   2) cov_pos_trace >= 19998              → Idem da covariance
#   3) inliers=0 ou inlier_ratio=0         → nenhuma correspondência visual encontrada naquele frame
COV_MAX       = 100

def _filter(df):
    return df[
        (df['cov_xx']        <  COV_MAX)       &
        (df['cov_yy']        <  COV_MAX)       &
        (df['cov_pos_trace'] <  COV_MAX)       &
        (df['inliers']       >  0)             &
        (df['inlier_ratio']  >  0)
    ].copy()

loc_2cam = _filter(loc_2cam_raw)
loc_1cam = _filter(loc_1cam_raw)

n_removed_2cam = len(loc_2cam_raw) - len(loc_2cam)
n_removed_1cam = len(loc_1cam_raw) - len(loc_1cam)
print(f'Duas câmeras : {len(loc_2cam_raw):5d} amostras brutas → {len(loc_2cam):5d} válidas ({n_removed_2cam:4d} removidas)')
print(f'Uma câmera   : {len(loc_1cam_raw):5d} amostras brutas → {len(loc_1cam):5d} válidas ({n_removed_1cam:4d} removidas)')

Duas câmeras :   217 amostras brutas →   129 válidas (  88 removidas)
Uma câmera   :   200 amostras brutas →   132 válidas (  68 removidas)


##### Validação da Distribuição — Antes de calcular medianas

Antes de prosseguir com as análises, é importante verificar a distribuição de cada métrica para identificar outliers remanescentes. O filtro do sentinel `cov=9999` remove localizações inválidas, mas podem existir valores extremos legítimos (ex: momentos de perda de localização parcial) que distorcem as medianas.

In [404]:
metrics = ['inliers', 'inlier_ratio', 'hypothesis_ratio', 'cov_pos_trace', 'detection_time_ms']

fig = make_subplots(
    rows=len(metrics), cols=2,
    column_titles=['Duas Câmeras', 'Uma Câmera'],
    row_titles=metrics,
    vertical_spacing=0.06
)

for row, metric in enumerate(metrics, start=1):
    fig.add_trace(go.Histogram(
        x=loc_2cam[metric], name=metric,
        marker_color='royalblue', opacity=0.7,
        showlegend=False
    ), row=row, col=1)
    fig.add_trace(go.Histogram(
        x=loc_1cam[metric], name=metric,
        marker_color='tomato', opacity=0.7,
        showlegend=False
    ), row=row, col=2)

fig.update_layout(
    title='Distribuição das Métricas — Duas Câmeras vs Uma Câmera',
    height=300 * len(metrics), width=900,
    hovermode='closest'
)
fig.show()

# Estatísticas descritivas
print('=== Duas Câmeras ===')
print(loc_2cam[metrics].describe().round(4).to_string())
print()
print('=== Uma Câmera ===')
print(loc_1cam[metrics].describe().round(4).to_string())

=== Duas Câmeras ===
        inliers  inlier_ratio  hypothesis_ratio  cov_pos_trace  detection_time_ms
count  129.0000      129.0000          129.0000       129.0000           129.0000
mean    35.7054        0.0903            0.6977         0.0184           398.8605
std     17.6951        0.0464            0.4611         0.0249            96.0611
min      3.0000        0.0064            0.0000         0.0003           156.3200
25%     27.0000        0.0696            0.0000         0.0044           335.8000
50%     38.0000        0.0959            1.0000         0.0091           385.1100
75%     47.0000        0.1246            1.0000         0.0299           449.2400
max     70.0000        0.1884            1.0000         0.2276           806.9000

=== Uma Câmera ===
        inliers  inlier_ratio  hypothesis_ratio  cov_pos_trace  detection_time_ms
count  132.0000      132.0000          132.0000       132.0000           132.0000
mean    54.8485        0.1464            0.4394         0

##### Por que remover frames com `inliers = 0` ou `inlier_ratio = 0`?

Os histogramas acima revelam uma concentração de frames com `inliers = 0` e `inlier_ratio = 0`. Esses valores não representam erros de medição, mas sim **frames em que o RTABMAP não encontrou nenhuma correspondência visual com o mapa**.

Isso ocorre em situações como:
- **Início da execução** — o mapa ainda não foi carregado completamente ou o robô está em posição não mapeada (ex: docking)
- **Perda momentânea de rastreamento** — movimento rápido, área sem textura ou oclusão
- **Transição entre espaços** — o robô entrou em região onde o mapa ainda não tem referências visuais suficientes

Nesses frames, as métricas derivadas (`inlier_ratio`, `hypothesis_ratio`, `cov_pos_trace`) ficam em estados degenerados e não refletem a qualidade da localização em regime estacionário. Incluí-los enviesaria as medianas e os boxplots para baixo, mascarando a real capacidade do sistema.

**Decisão:** remover todos os frames com `inliers == 0` ou `inlier_ratio == 0` de ambos os grupos antes de qualquer análise comparativa.

#### 3.2 Inlier Ratio

Proporção de correspondências visuais válidas em relação ao total de matches. Valor alto indica que o RTABMAP encontrou muitos pontos consistentes com o mapa — sinal de localização confiável. Duas câmeras deveria apresentar valores maiores por ter maior campo de visão.

In [405]:
loc_2cam = loc_2cam_raw[
    (loc_2cam_raw['inliers'] > 0) &
    (loc_2cam_raw['inlier_ratio'] > 0)
].copy()
loc_1cam = loc_1cam_raw[
    (loc_1cam_raw['inliers'] > 0) &
    (loc_1cam_raw['inlier_ratio'] > 0)
].copy()


fig = go.Figure()
fig.add_trace(go.Box(
    y=loc_2cam['inlier_ratio'], name='Duas câmeras',
    marker_color='royalblue', boxpoints='outliers'
))
fig.add_trace(go.Box(
    y=loc_1cam['inlier_ratio'], name='Uma câmera',
    marker_color='tomato', boxpoints='outliers'
))
fig.update_layout(
    title='Inlier Ratio — Duas Câmeras vs Uma Câmera',
    yaxis_title='Inlier Ratio', hovermode='closest'
)
fig.show()

print(f'Mediana inlier_ratio — duas câmeras : {loc_2cam["inlier_ratio"].median():.4f}')
print(f'Mediana inlier_ratio — uma câmera   : {loc_1cam["inlier_ratio"].median():.4f}')

Mediana inlier_ratio — duas câmeras : 0.0959
Mediana inlier_ratio — uma câmera   : 0.1528


#### 3.3 Hypothesis Ratio

O `hypothesis_ratio` é calculado pelo RTABMAP como:

$$\text{hypothesis\_ratio} = \frac{\text{score da melhor hipótese}}{\sum \text{score de todas as hipóteses}}$$

**Interpretação:**
- **Próximo de 1** → uma hipótese domina completamente — o RTABMAP está confiante de onde o robô está
- **Próximo de 0** → várias hipóteses competindo com pesos similares — localização ambígua
- O RTABMAP usa um limiar interno (tipicamente **0.36**) para aceitar uma hipótese como loop closure confirmado

**O que esperar na distribuição:** o dado costuma ser **bimodal** — frames onde o sistema reconhece o lugar com clareza (valores altos) e frames em transição ou ambíguos (valores baixos). Antes de comparar os grupos, vamos explorar essa estrutura.

In [406]:
# Total de frames com hypothesis_ratio = 1 para cada grupo
total_2cam = len(loc_2cam)
total_1cam = len(loc_1cam)
count_one_2cam = (loc_2cam['hypothesis_ratio'] == 1).sum()
count_one_1cam = (loc_1cam['hypothesis_ratio'] == 1).sum()
pct_one_2cam = count_one_2cam / total_2cam * 100
pct_one_1cam = count_one_1cam / total_1cam * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Total de frames (ratio = 1)', '% do total de frames']
)

colors = ['royalblue', 'tomato']
groups = ['Duas câmeras', 'Uma câmera']

fig.add_trace(go.Bar(
    x=groups, y=[count_one_2cam, count_one_1cam],
    marker_color=colors,
    text=[str(count_one_2cam), str(count_one_1cam)],
    textposition='outside',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=groups, y=[pct_one_2cam, pct_one_1cam],
    marker_color=colors,
    text=[f'{pct_one_2cam:.1f}%', f'{pct_one_1cam:.1f}%'],
    textposition='outside',
    showlegend=False
), row=1, col=2)

fig.update_yaxes(title_text='Frames', row=1, col=1)
fig.update_yaxes(title_text='%', range=[0, 100], row=1, col=2)
fig.update_layout(
    title='Comparação Total — Frames com Hypothesis Ratio = 1',
    hovermode='closest', width=800
)
fig.show()

print(f'Duas câmeras : {count_one_2cam:4d} / {total_2cam} frames com ratio=1  ({pct_one_2cam:.1f}%)')
print(f'Uma câmera   : {count_one_1cam:4d} / {total_1cam} frames com ratio=1  ({pct_one_1cam:.1f}%)')
melhor = 'Duas câmeras' if pct_one_2cam > pct_one_1cam else 'Uma câmera'
print(f'\n→ {melhor} obteve hipótese confirmada em maior proporção dos frames.')

Duas câmeras :   90 / 129 frames com ratio=1  (69.8%)
Uma câmera   :   58 / 132 frames com ratio=1  (43.9%)

→ Duas câmeras obteve hipótese confirmada em maior proporção dos frames.


In [407]:
# Proporção de hypothesis_ratio = 1 por run (variabilidade entre execuções)
pct_per_run_2cam = (
    loc_2cam.groupby('run')['hypothesis_ratio']
    .apply(lambda s: (s == 1).mean() * 100)
    .reset_index(name='pct_one')
)
pct_per_run_1cam = (
    loc_1cam.groupby('run')['hypothesis_ratio']
    .apply(lambda s: (s == 1).mean() * 100)
    .reset_index(name='pct_one')
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=pct_per_run_2cam['run'], y=pct_per_run_2cam['pct_one'],
    name='Duas câmeras', marker_color='royalblue', opacity=0.8
))
fig.add_trace(go.Bar(
    x=pct_per_run_1cam['run'], y=pct_per_run_1cam['pct_one'],
    name='Uma câmera', marker_color='tomato', opacity=0.8
))
fig.update_layout(
    title='% de Frames com Hypothesis Ratio = 1 por Run',
    xaxis_title='Run', yaxis_title='% de Frames (ratio=1)',
    barmode='group', hovermode='closest', width=1000
)
fig.show()

print(f'Mediana % ratio=1 — duas câmeras : {pct_per_run_2cam["pct_one"].median():.1f}%')
print(f'Mediana % ratio=1 — uma câmera   : {pct_per_run_1cam["pct_one"].median():.1f}%')

Mediana % ratio=1 — duas câmeras : 69.2%
Mediana % ratio=1 — uma câmera   : 42.3%


#### 3.4 Covariância Posicional (cov_pos_trace)

O traço da covariância posicional (`cov_xx + cov_yy`) representa a incerteza total na estimativa de posição. Valores menores indicam que o filtro está mais confiante na localização.

In [408]:
loc_2cam = _filter(loc_2cam_raw)
loc_1cam = _filter(loc_1cam_raw)


fig = go.Figure()
fig.add_trace(go.Box(
    y=loc_2cam['cov_pos_trace'], name='Duas câmeras',
    marker_color='royalblue', boxpoints='outliers'
))
fig.add_trace(go.Box(
    y=loc_1cam['cov_pos_trace'], name='Uma câmera',
    marker_color='tomato', boxpoints='outliers'
))
fig.update_layout(
    title='Covariância Posicional (cov_pos_trace) — Duas Câmeras vs Uma Câmera',
    yaxis_title='cov_pos_trace', hovermode='closest'
)
fig.show()

print(f'Mediana cov_pos_trace — duas câmeras : {loc_2cam["cov_pos_trace"].median():.6f}')
print(f'Mediana cov_pos_trace — uma câmera   : {loc_1cam["cov_pos_trace"].median():.6f}')
loc_2cam["cov_pos_trace"].max()

Mediana cov_pos_trace — duas câmeras : 0.009146
Mediana cov_pos_trace — uma câmera   : 0.004875


np.float64(0.227565)

In [409]:
col = 'cov_pos_trace'

x_min = min(loc_2cam[col].min(), loc_1cam[col].min())
x_max = max(loc_2cam[col].max(), loc_1cam[col].max())

fig = make_subplots(rows=1, cols=2, column_titles=['Duas Câmeras', 'Uma Câmera'],
                    shared_xaxes=True, shared_yaxes=True)

fig.add_trace(go.Histogram(
    x=loc_2cam[col], marker_color='royalblue', opacity=0.7, showlegend=False,
    xbins=dict(start=x_min, end=x_max, size=(x_max - x_min) / 100)
), row=1, col=1)
fig.add_trace(go.Histogram(
    x=loc_1cam[col], marker_color='tomato', opacity=0.7, showlegend=False,
    xbins=dict(start=x_min, end=x_max, size=(x_max - x_min) / 100)
), row=1, col=2)

fig.update_xaxes(title_text=col, range=[x_min, x_max])
fig.update_yaxes(title_text='Frames', row=1, col=1)
fig.update_layout(
    title='Distribuição da Covariância Posicional (cov_pos_trace)',
    height=400, width=900, hovermode='closest'
)
fig.show()

##### Mediana da Covariância por Run

O histograma mostra a distribuição agregada de todos os frames, mas não revela se o comportamento é consistente entre execuções. A mediana da `cov_pos_trace` calculada por run permite verificar se há runs com incerteza sistematicamente maior — o que indicaria problemas pontuais de localização em certas execuções, e não apenas variação natural dentro de uma run.

Valores mais baixos indicam que o RTABMAP estava mais confiante na estimativa de posição ao longo daquela execução.

In [410]:
col = 'cov_pos_trace'

med_2cam_run = loc_2cam.groupby('run')[col].median().reset_index(name='mediana')
med_1cam_run = loc_1cam.groupby('run')[col].median().reset_index(name='mediana')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=med_2cam_run['run'], y=med_2cam_run['mediana'],
    name='Duas câmeras', marker_color='royalblue', opacity=0.8
))
fig.add_trace(go.Bar(
    x=med_1cam_run['run'], y=med_1cam_run['mediana'],
    name='Uma câmera', marker_color='tomato', opacity=0.8
))
fig.update_layout(
    title='Mediana da Covariância Posicional (cov_pos_trace) por Run',
    xaxis_title='Run', yaxis_title='Mediana cov_pos_trace',
    barmode='group', hovermode='closest', width=1000
)
fig.show()

print(f'Mediana global — duas câmeras : {med_2cam_run["mediana"].median():.6f}')
print(f'Mediana global — uma câmera   : {med_1cam_run["mediana"].median():.6f}')
melhor = 'Duas câmeras' if med_2cam_run['mediana'].median() < med_1cam_run['mediana'].median() else 'Uma câmera'
print(f'\n→ {melhor} apresentou menor incerteza mediana por run.')

Mediana global — duas câmeras : 0.009486
Mediana global — uma câmera   : 0.005442

→ Uma câmera apresentou menor incerteza mediana por run.


#### 3.5 Tempo de Detecção

Custo computacional do ciclo de localização. Duas câmeras processa mais dados visuais — espera-se tempo maior. O `detection_time_ms` é o tempo da etapa de reconhecimento visual; o `total_time_ms` inclui todas as etapas do RTABMAP.

In [411]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['detection_time_ms', 'total_time_ms'])

for col_idx, col in enumerate(['detection_time_ms', 'total_time_ms'], start=1):
    fig.add_trace(go.Box(
        y=loc_2cam[col], name='Duas câmeras',
        marker_color='royalblue', boxpoints='outliers',
        legendgroup='2cam', showlegend=(col_idx == 1)
    ), row=1, col=col_idx)
    fig.add_trace(go.Box(
        y=loc_1cam[col], name='Uma câmera',
        marker_color='tomato', boxpoints='outliers',
        legendgroup='1cam', showlegend=(col_idx == 1)
    ), row=1, col=col_idx)

fig.update_yaxes(title_text='ms')
fig.update_layout(title='Tempo de Processamento — Duas Câmeras vs Uma Câmera', hovermode='closest', width=900)
fig.show()

print(f'Mediana detection_time_ms — duas câmeras : {loc_2cam["detection_time_ms"].median():.1f} ms')
print(f'Mediana detection_time_ms — uma câmera   : {loc_1cam["detection_time_ms"].median():.1f} ms')

Mediana detection_time_ms — duas câmeras : 385.1 ms
Mediana detection_time_ms — uma câmera   : 252.4 ms
